# PAP CAD Colab Workbench - Iteration 47 / CAD v1.2.0

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.**  
**AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.**  
**Package license: GPL-3.0-only.** Third-party backends retain their own licenses; see `COPYRIGHT-AND-LICENSING.md` inside the uploaded package.

This notebook is an optional remote browser workbench for the Physical AI Platform CAD drop-in. It is not a replacement CAD kernel, canonical store, product runtime, safety authority or evidence authority. It does not make a local CAD application mandatory.

**Notebook purpose:** upload the bounded CAD package, install optional remote-runtime dependencies, execute package tests and samples, inspect backend availability, and optionally demonstrate GPU broad-phase candidate filtering.

**Important:** a Colab GPU does not automatically accelerate OCCT B-rep Booleans, CGAL exact-kernel operations or Manifold3D. The GPU cell below demonstrates AABB broad-phase screening only. It does not compute or certify a Boolean intersection.

## Open in Colab

Open https://colab.research.google.com/ and use **File → Upload notebook** to upload this `.ipynb`. CPU runtime is sufficient for the package tests. GPU is optional for the broad-phase demonstration.

## 0. Runtime configuration

Set flags before running. Installation cells are optional and may take several minutes. Colab sessions are ephemeral; download reports before closing the session.

In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.
# AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# GPL-3.0-only for this package; see package/COPYRIGHT-AND-LICENSING.md.

ENABLE_OCCT = True
ENABLE_MANIFOLD3D = False
ENABLE_CGAL_BUILD = False  # Requires apt packages/compiler; can take several minutes.
ENABLE_GPU_DEMO = True    # PyTorch broad-phase; CUDA if available, otherwise CPU fallback.
ENABLE_WARP_DEMO = False  # Optional NVIDIA Warp geometry-query kernel; install in Colab only.
RUN_FULL_PACKAGE_TESTS = True

import os, sys, json, platform, pathlib, subprocess, textwrap, hashlib, time
print({
    "python": sys.version,
    "platform": platform.platform(),
    "colab": os.path.exists("/content"),
    "gpu_requested": ENABLE_GPU_DEMO,
})

## 1. Upload and verify the CAD package ZIP

Upload the `cad.zip` handoff archive. The notebook extracts only the `cad/` package into the remote Colab runtime.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_candidates:
    raise RuntimeError("Upload the CAD package ZIP (cad.zip).")
CAD_ZIP = zip_candidates[0]
zip_sha256 = hashlib.sha256(pathlib.Path(CAD_ZIP).read_bytes()).hexdigest()
print("Uploaded:", CAD_ZIP, "SHA256:", zip_sha256)

import zipfile, shutil
WORK = pathlib.Path('/content/pap-cad-workbench')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(CAD_ZIP) as zf:
    names = zf.namelist()
    if not any(n.endswith('manifest.json') and '/cad/' in n for n in names):
        raise RuntimeError("ZIP does not appear to contain the expected cad/ package.")
    zf.extractall(WORK)
CAD_DIR = next(p.parent for p in WORK.rglob('manifest.json') if p.parent.name == 'cad')
manifest = json.loads((CAD_DIR/'manifest.json').read_text())
print("Package:", manifest.get('package'), "version:", manifest.get('version'))
print("CAD directory:", CAD_DIR)

## 2. Runtime and dependency setup

Node is used for browser-safe JavaScript tests and the registry. OCP is the Python binding used by the OCCT backend. Installations occur in Colab only, not on the user's computer.

In [ ]:
# Colab normally provides Node. Install it only if absent.
node = shutil.which('node')
if node is None:
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','nodejs','npm'], check=True)
    node = shutil.which('node')
print("Node:", subprocess.check_output([node,'--version'], text=True).strip())

if ENABLE_OCCT:
    subprocess.run([sys.executable,'-m','pip','install','-q','cadquery-ocp'], check=True)
    import OCP
    print("OCP import PASS", getattr(OCP,'__version__','version not exposed'))

if ENABLE_MANIFOLD3D:
    subprocess.run([sys.executable,'-m','pip','install','-q','manifold3d'], check=True)
    import manifold3d
    print("Manifold3D import PASS", getattr(manifold3d,'__version__','version not exposed'))

## 3. Execute package tests

This executes the package's own test script. A failed command stops the cell. Optional backend absence is reported by the package as unavailable; it is not counted as a backend pass.

In [ ]:
if RUN_FULL_PACKAGE_TESTS:
    proc = subprocess.run(['bash','run-tests.sh'], cwd=CAD_DIR, text=True, capture_output=True)
    print(proc.stdout)
    if proc.stderr: print("STDERR:\n", proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"CAD package tests failed with exit code {proc.returncode}")
    print("PACKAGE TEST SCRIPT: PASS")

## 4. Run native OCCT samples and capture machine-readable outputs

The sample files include analytic expected volumes. The OCCT backend is a host-side engine executed inside the remote notebook runtime. The notebook checks outcome, volume and topology fields.

In [ ]:
def run_occt_sample(sample_path):
    sample = json.loads(pathlib.Path(sample_path).read_text())
    req = {
        'base': sample['base'], 'cutter': sample['cutter'],
        'tolerance': sample.get('tolerance', 1e-7),
        'deflection': sample.get('deflection', 1e-3),
        'angularDeflection': sample.get('angularDeflection', 0.25),
    }
    p = subprocess.run([sys.executable, str(CAD_DIR/'backends'/'occt_backend.py')],
                       input=json.dumps(req), text=True, capture_output=True)
    if p.returncode != 0: raise RuntimeError(p.stderr or p.stdout)
    result = json.loads(p.stdout)
    expected = sample.get('expectedIntersectionVolume')
    if expected is None:
        assert result.get('outcome') == 'HOLD', (sample['name'], result)
    else:
        assert result.get('outcome') == 'OK', (sample['name'], result)
        assert abs(result['volume']-expected) <= sample.get('volumeTolerance', 1e-6), (sample['name'], result['volume'], expected)
        assert result.get('topology',{}).get('watertight') is True
    return {'sample': sample['name'], 'outcome': result.get('outcome'),
            'volume': result.get('volume'), 'expected': expected,
            'engine': result.get('engine'), 'code': result.get('code')}

if ENABLE_OCCT:
    sample_rows=[]
    for sample_file in sorted((CAD_DIR/'samples').glob('*.json')):
        sample_rows.append(run_occt_sample(sample_file))
    from pprint import pprint
    pprint(sample_rows)
    print("OCCT SAMPLE MATRIX: PASS")
else:
    print("OCCT disabled; native samples not executed")

## 5. Optional CGAL EPECK build

The source bridge uses CGAL exact-predicate/exact-construction geometry. This is optional and not required for the browser product. Enable only in a Colab session where package installation is permitted. A successful compile is not enough: run the executable against the same analytic samples and compare its outputs before enabling it in a release.

In [ ]:
if ENABLE_CGAL_BUILD:
    apt = ['apt-get','install','-y','cmake','g++','libcgal-dev','libgmp-dev','libmpfr-dev','nlohmann-json3-dev']
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(apt, check=True)
    build = CAD_DIR/'backends'/'build-colab'
    subprocess.run(['cmake','-S',str(CAD_DIR/'backends'),'-B',str(build),'-DCMAKE_BUILD_TYPE=Release'], check=True)
    subprocess.run(['cmake','--build',str(build),'--config','Release','-j2'], check=True)
    print("CGAL bridge compiled:", build)
else:
    print("CGAL build skipped by configuration. Source is not runtime-qualified by this skip.")

## 6. Optional GPU broad-phase demonstration

This cell uses PyTorch to batch AABB overlap tests. It demonstrates a legitimate GPU use: quickly filtering candidate pairs before an expensive geometric operation. It does not calculate mesh intersections, does not replace OCCT/CGAL/Manifold, and must never be used as canonical geometric proof.

In [ ]:
if ENABLE_GPU_DEMO:
    try:
        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print('PyTorch:', torch.__version__, 'device:', device)
        if device == 'cuda': print('GPU:', torch.cuda.get_device_name(0))
        # Synthetic AABBs; replace with bounds computed from validated input meshes.
        torch.manual_seed(47)
        n = 100_000
        a_min = torch.rand((n,3), device=device) * 100
        a_size = torch.rand((n,3), device=device) * 5 + 0.01
        b_min = torch.rand((n,3), device=device) * 100
        b_size = torch.rand((n,3), device=device) * 5 + 0.01
        a_max, b_max = a_min + a_size, b_min + b_size
        overlap = ((a_min <= b_max) & (b_min <= a_max)).all(dim=1)
        count = int(overlap.sum().item())
        print({'pairs': n, 'candidate_overlaps': count, 'device': device,
               'meaning': 'broad-phase candidates only; narrow-phase Boolean still required'})
    except Exception as exc:
        print('GPU demonstration unavailable:', repr(exc))
else:
    print('GPU demo disabled')

## 6b. Optional NVIDIA Warp geometry-kernel path

NVIDIA Warp is a GPU/CPU kernel framework with mesh query primitives. This optional cell compiles a small AABB broad-phase kernel using geometry bounds derived from the included box-overlap fixture. It is a candidate-pair filter only. Warp is not presented here as an exact Boolean engine. Enable `ENABLE_WARP_DEMO` in the configuration cell if you want to install it in the remote runtime. Warp's CUDA wheel/driver compatibility varies by release; a CPU fallback is retained.

In [ ]:
if ENABLE_WARP_DEMO:
    subprocess.run([sys.executable,'-m','pip','install','-q','warp-lang'], check=True)
    import warp as wp
    wp.init()
    cuda_count = wp.get_cuda_device_count()
    warp_device = 'cuda:0' if cuda_count > 0 else 'cpu'
    print('Warp:', getattr(wp,'__version__','version not exposed'), 'device:', warp_device)

    @wp.kernel
    def aabb_overlap_kernel(a_min: wp.array(dtype=wp.vec3),
                            a_max: wp.array(dtype=wp.vec3),
                            b_min: wp.array(dtype=wp.vec3),
                            b_max: wp.array(dtype=wp.vec3),
                            out: wp.array(dtype=wp.int32)):
        i = wp.tid()
        amin, amax = a_min[i], a_max[i]
        bmin, bmax_i = b_min[i], b_max[i]
        hit = (amin[0] <= bmax_i[0] and bmin[0] <= amax[0] and
               amin[1] <= bmax_i[1] and bmin[1] <= amax[1] and
               amin[2] <= bmax_i[2] and bmin[2] <= amax[2])
        out[i] = 1 if hit else 0

    fixture = json.loads((CAD_DIR/'samples'/'box-overlap.json').read_text())
    def bounds_of(mesh):
        pts = mesh['vertices']
        return ([min(p[k] for p in pts) for k in ('x','y','z')],
                [max(p[k] for p in pts) for k in ('x','y','z')])
    amin, amax = bounds_of(fixture['base'])
    bmin0, bmax0 = bounds_of(fixture['cutter'])
    count = 100_000
    a0, a1, b0, b1 = [], [], [], []
    for i in range(count):
        dx = ((i % 101) - 50) * 0.15
        dy = (((i // 101) % 101) - 50) * 0.15
        dz = (((i // 10201) % 17) - 8) * 0.15
        a0.append(wp.vec3(*amin)); a1.append(wp.vec3(*amax))
        b0.append(wp.vec3(bmin0[0]+dx,bmin0[1]+dy,bmin0[2]+dz))
        b1.append(wp.vec3(bmax0[0]+dx,bmax0[1]+dy,bmax0[2]+dz))
    aa0 = wp.array(a0,dtype=wp.vec3,device=warp_device)
    aa1 = wp.array(a1,dtype=wp.vec3,device=warp_device)
    bb0 = wp.array(b0,dtype=wp.vec3,device=warp_device)
    bb1 = wp.array(b1,dtype=wp.vec3,device=warp_device)
    out = wp.zeros(count,dtype=wp.int32,device=warp_device)
    wp.launch(aabb_overlap_kernel,dim=count,inputs=[aa0,aa1,bb0,bb1,out],device=warp_device)
    wp.synchronize_device(warp_device)
    print({'pairs_screened':count,'candidate_pairs':int(out.numpy().sum()),
           'device':warp_device,'status':'broad-phase only; exact/narrow-phase Boolean still required'})
else:
    print('Warp demo disabled; no additional dependency installed.')

## 7. Backend discovery and release report

The registry probes the local runtime. Availability is not the same as qualification: the report retains explicit statuses for optional backends.

In [ ]:
registry = CAD_DIR/'backends'/'registry.mjs'
script = f"""import {{ discoverCadBackends }} from {json.dumps(registry.as_uri())};
const r = await discoverCadBackends();
console.log(JSON.stringify(r.map(x => ({{name:x.name,kind:x.kind,available:x.available,detail:x.detail}})), null, 2));
"""
# Write a temporary .mjs alongside the package to ensure relative module imports resolve.
probe_file = CAD_DIR/'colab-backend-probe.mjs'
probe_file.write_text(script)
probe = subprocess.run([node,str(probe_file)],text=True,capture_output=True)
print(probe.stdout)
if probe.returncode != 0: print(probe.stderr)
probe_file.unlink(missing_ok=True)

report = {
  'copyright': 'Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.',
  'trademarks': 'AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.',
  'package': manifest.get('package'), 'version': manifest.get('version'),
  'zip_sha256': zip_sha256, 'python': sys.version,
  'platform': platform.platform(), 'node': subprocess.check_output([node,'--version'],text=True).strip(),
  'gpu': (torch.cuda.get_device_name(0) if 'torch' in globals() and torch.cuda.is_available() else 'not available / not enabled'),
  'note': 'Optional backend availability is not a qualification claim.'
}
report_path = pathlib.Path('/content/cad-colab-verification-report.json')
report_path.write_text(json.dumps(report,indent=2)+'\n')
print(json.dumps(report,indent=2))

## 8. Download verification report

Download the report before the Colab runtime expires. Keep it with the exact CAD ZIP and any backend build logs. Do not treat this report as canonical platform evidence until Grok's integration adapter binds it to canonical input IDs/revisions and validates the output.

In [ ]:
from google.colab import files
files.download(str(report_path))

## 9. Interpretation and limits

- If package tests pass, the package test suite passed in this runtime. It does not imply every optional backend was installed.
- If OCCT samples pass, the tested sample geometries passed the OCCT adapter and analytic-volume checks.
- If CGAL is skipped or fails to compile, CGAL remains unverified.
- If Manifold3D is disabled or unavailable, it remains unverified.
- GPU AABB overlap is only a broad-phase candidate filter.
- Colab is an optional remote runtime. The product remains browser-first and the canonical SIR/store remain in the platform.
- No test in this notebook establishes industrial certification, safety qualification, or physical verification.

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.**  
**AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.**